In [17]:
import polars as pl 
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
import plotly.express as px 
import plotly.graph_objects as go 

In [19]:
df = pd.read_csv("full-catalog.csv")
df.head(10)

,Course ID,Subject code,Catalog Number,Offering Unit,Course Title,Course Description,Min Units,Max Units,Repeatable for Credit,Total Completions Allowed,Total Units Allowed,Grading Basis,Components,Course Attributes,Enrollment Requirements,Course Requisites
0,43425,AREC,340,"Geography & Dev, Sch of",Food Entrepreneurship & Social Innovation,From food trucks to food banks... from new foo...,3.0,3.0,No,1,3.0,"GRD - Regular Grades A, B, C, D, E",Lecture,CE - CL (Cross Listed),-,-
1,39477,ABBS,696C,Graduate Interdisciplinary Prg,Arizona Biological and Biomedical Sciences Stu...,The development and exchange of scholarly info...,1.0,1.0,No,1,1.0,"GRD - Regular Grades A, B, C, D, E",Colloquium,-,-,Restricted to ABBS Doctoral Students
2,42678,ABBS,792,Graduate Interdisciplinary Prg,Directed Rotation Research,This course is designed to aid ABBS first year...,1.0,4.0,Yes,10,10.0,"GRD - Regular Grades A, B, C, D, E",Independent Study,-,-,Students should be enrolled in the ABBS Doctor...
3,40839,ABS,572,"Neuroscience, GIDP",Neurodevelopment in Action: How the brain is b...,Neurodevelopment provides a context in which t...,4.0,4.0,No,1,4.0,"GRD - Regular Grades A, B, C, D, E",Lecture,"GIDP - ABS (Applied Biosciences), GIDP - NRSC ...",-,One prior course in molecular cellular biology...
4,36183,ABS,593A,"Applied Biosciences, GIDP",Internship in Applied Biosciences,Specialized work or service on an individual b...,1.0,9.0,Yes,99,999.0,"SPF - Alternative Grading S, P, F",Independent Study,-,-,-
5,36996,ABS,599,"Applied Biosciences, GIDP",Independent Study,Qualified students working on an individual ba...,1.0,6.0,Yes,99,999.0,"SPF - Alternative Grading S, P, F",Independent Study,-,-,-
6,36995,ABS,909,"Applied Biosciences, GIDP",Master's Report,Independent study or special project or formal...,1.0,6.0,Yes,99,999.0,"SPF - Alternative Grading S, P, F",Independent Study,-,-,-
7,35261,ACBS,102L,Animal & Biomedical Sciences,Introduction to Animal Science Laboratory,This course will expose students to various pr...,1.0,1.0,No,1,1.0,"GRD - Regular Grades A, B, C, D, E",Laboratory,-,-,-
8,35243,ACBS,102R,Animal & Biomedical Sciences,Introduction to Animal Science,This course is a comprehensive review of the l...,3.0,3.0,No,1,3.0,"GRD - Regular Grades A, B, C, D, E",Lecture,-,-,-
9,6627,ACBS,142,Animal & Biomedical Sciences,Introduction to Animal Racing Industry,"Overview of the history, terminology, personne...",2.0,2.0,No,1,2.0,"GRD - Regular Grades A, B, C, D, E",Lecture,-,-,-


In [20]:
import pandas as pd
import re

# 1. Load the original full catalog
# Note: make sure 'full-catalog.csv' is in the same directory as this script
catalog_df = pd.read_csv('full-catalog.csv')

# 2. The raw text of courses you provided
raw_courses_text = """
ECON 150C1 An Economics Perspective *
ECON 150C2 Climate Science and Economics: How Should Policy Control Warming? *
ECON 200 Basic Economic Issues *
ECON 205 The Ethics of Economics of Wealth Creation (also PHIL 205)
ECON 291 Preceptorship *
ECON 299 Independent Study *
ECON 299H Honors Independent Study *
ECON 300 Microeconomic Analysis for Business Decisions *
ECON 301 Microeconomic Analysis and Applications*
ECON 303 Economics of Crime
ECON 307 Economic History of the United States
ECON 308 World Economic History
ECON 309 European Economic History until the Industrial Revolution
ECON 313 Economics of Futures Markets
ECON 323 Economics of Sports
ECON 325 Historical Development of Financial and Economic Institutions
ECON 330 Macroeconomic and Global Institutions and Policy *
ECON 331 Macroeconomic Analysis and Policy *
ECON 332 Intermediate Macroeconomics *
ECON 337 The Economics of Politics and Policymaking
ECON 338 Law and Economics
ECON 339 Economic Statistics *
ECON 340 International Economics and Policy
ECON 342 Economic of Latin America
ECON 361 Intermediate Microeconomics *
ECON 370 China’s Economic Development
ECON 371 Economic Development
ECON 373 Environmental Economics
ECON 382 Labor and Public Policy
ECON 391 Preceptorship **
ECON 393 Internship **
ECON 396H Honors Proseminar **
ECON 399 Independent Study **
ECON 399H Honors Independent Study **
ECON 400 Economic Strategy for Business Decisions*
ECON 405 Economic Models of Discrimination
ECON 406 Introduction to Experimental Economics
ECON 407 Economics of Strategy
ECON 410 Topics in Economics Analysis
ECON 418 Introduction to Econometrics
ECON 420 Intro to Math Economics
ECON 422 Introduction to Health Economics
ECON 425 Topics in the Economic History of the United States
ECON 426 Economic Foundations for Financial Markets
ECON 427 Current Topics in Healthcare Economics and Policy
ECON 430 Monetary Economics
ECON 431 Games and Decisions
ECON 435 Public Sector Economics
ECON 436 Behavioral Economics
ECON 439 Law, Statistics and Economics
ECON 440 Behavioral Game Theory
ECON 442 International Macroeconomics
ECON 443 International Trade Theory
ECON 452 Informational Economics and the Internet
ECON 453 Quantitative Methods for Economic Strategy
ECON 454 Advanced Data Analytics and Modeling: Advanced Quantitative Analysis for Economic Strategy
ECON 460 Industrial Organization
ECON 461 Economics of Regulated Industries
ECON 462 Firms, Markets and Competition
ECON 473 Energy Markets and Environmental Economics
ECON 478 Economics of the Natural Environment
ECON 481 Economics of Wage Determination
ECON 482 Labor and the Economy
FIN 150C Finance and Society: The Good, The Bad and The Ugly
FIN 302 Personal Investing
FIN 303 Small Business Finance
FIN 304 Real Estate Principles
FIN 305 Financial Management: Applied Methods and Tools
FIN 311 Introduction to Finance
FIN 360 Quantitative Financial Management
FIN 360L Quantitative Financial Management Techniques
FIN 397 Workshop (Applied Excel Modeling)
FIN 391 Preceptorship
FIN 401 Financial Analysis with Bloomberg Certification *
FIN 412 Corporate Financial Problems
FIN 413 Financial Modeling *
FIN 414 International Finance *
FIN 415 Corporate Strategy in International Finance *
FIN 417 Wall Street Professional Development
FIN 418 Private Equity and Venture Capital*
FIN 421 Investments
FIN 422 Risk Management and Derivatives *
FIN 423A Applied Investment Management *
FIN 423B Applied Investment Management *
FIN 429 Current Events in Capital Markets*
FIN 430 Topical Issues in Finance*
FIN 431 Financial Intermediaries *
FIN 460 Real Estate Finance and Investment *
FIN 480 Finance for New Ventures
FIN 493 Internship
FIN 496 Special Topics in Finance *
MGMT 202 Ethics Issues in Business
MGMT 291 Preceptorship
MGMT 299 Independent Study
MGMT 299H Honors Independent Study
MGMT 308 Augustus of Empire
MGMT 310A Organization Behavior and Management
MGMT 310D Leadership and Management Skills Development
MGMT 330 Introduction to Human Resources Management
MGMT 336 Leadership in Organizations
MGMT 351 Sports Administration and Planning
MGMT 352 Sport Tourism and Event Management
MGMT 353 Sports Negotiation and Compliance
MGMT 354 The Business of College Sports
MGMT 355 Sports Marketing Management
MGMT 356 Sports Communication
MGMT 357 The Lifecycle of Elite Athletes / Life During and After Sports
MGMT 359 Special Topics in Sports Management
MGMT 380 Social Innovation Organizations
MGMT 381 Management of Effective Non Profit Organizations
MGMT 382 Nonprofit Consulting
MGMT 391 Preceptorship
MGMT 394 Practicum
MGMT 396H Management Honors Special Topics Seminar
MGMT 399 Independent Study
MGMT 399H Honors Independent Study
MGMT 402 Integrating Business Fundamentals with Ethics and Law in Management
MGMT 407 Business Environment in the Middle East and North Africa
MGMT 408 Compensation
MGMT 420 Business Law
MGMT 428 Healthcare Consulting Skills
MGMT 430 Human Resources Policy
MGMT 431 Human Resources Management in Services
MGMT 432A Applied Topics in Bargaining & Negotiations
MGMT 432B Bargaining and Negotiation in Organizations
MGMT 433 Strategic Human Resource Management
MGMT 434 Health Care Organization and Management
MGMT 435 International Management
MGMT 437 Organizational Staffing
MGMT 438 Health Care Organization and Management
MGMT 440 Leadership in a Complex World
MGMT 442 Designing and Managing Organizations
MGMT 444 Managing Groups and Teams
MGMT 445 Interactive Behavior in Small Groups
MGMT 448 Healthcare Entrepreneurship
MGMT 450 Training and Development
Communication and Organizational Change
MGMT 454 Management of Sales Operations
MGMT 455 Exploring Management Problems in the Lab
MGMT 458 Conflict and Cooperation in the Dyad
MGMT 460 Human Resource Information Systems
MGMT 463 Doing Business In /With Africa: A Cultural Perspective
MGMT 465 Managing for Quality Improvement
MGMT 468 Persuasion in Entrepreneurial Contexts
MGMT 471 Strategic Management
MGMT 473A Production and Operations Management
MGMT 473B Production and Operations Management
MGMT 475 Topics in Management
MGMT 477 The Supply Chain and Logistics
MGMT 478 Building a High Performance Company
MGMT 480 Gender, Leadership and Organization
MGMT 483 Marketing Planning and Operational Decision-Making
MGMT 485 Decision Analysis
MGMT 486 Managerial Judgment and Decision
MGMT 488 Social Entrepreneurship
MGMT 491 Preceptorship
MGMT 493 Internship
MGMT 493L Legislative Internship
MGMT 496H Honors Seminar
MGMT 496Z University Management
MIS 111 Computers and the Internetworked Society
MIS 301 Data Structures and Algorithms
MIS 304 Using and Managing Information Systems
MIS 307 Business Data Communications
MIS 331 Database Management Systems
MIS 341 Information Systems Analysis and Design
MIS 373 Basic Operations Management
MIS 406 Healthcare Information Systems *
MIS 411 Social and Ethical Issues of the Internet *
MIS 415 Information Security in Public and Private Sectors *
MIS 416 Information Security Risk Management *
MIS 417 Systems Security Management *
MIS 427 Introduction to Enterprise Computing Environments *
MIS 429 Detection of Deception and Intent *
MIS 441 Information Systems Analysis and Design
MIS 464 Data Analytics*
MIS 465 Managing for Quality Improvement *
MIS 478 Project Management *
MIS 496A-002 Business Intelligence: Web and Social Media Analytics *
MIS 498 Senior Capstone
MIS 498H Honors Thesis *
OSCM 471/571 Optimization and Decision Support Modeling for Business
OSCM 474/574 - Manufacturing Operations Management
OSCM 475/575 - Service Operations Management
OSCM 477/577 - Supply Chain and Logistics Management
MKTG 355 Sports Marketing Management
MKTG 361 Introduction to Marketing
MKTG 376 Marketing Analytics
MKTG 423 Digital Marketing *
MKTG 425 Advertising Management *
MKTG 426 Pricing and Channels *
MKTG 428 Sales Communication *
MKTG 430 Retail Marketing
MKTG 440 Marketing Research
MKTG 450 Buyer Behavior
MKTG 452 Integrated Marketing Communications *
MKTG 454 Management of Sales Operations *
MKTG 456 International Marketing Management *
MKTG 458 Healthcare Marketing *
MKTG 459 Product Management *
MKTG 460 Brand Management *
MKTG 471 Marketing Policies and Operations
MKTG 480 Marketing Research for Entrepreneurs
MKTG 498 American Marketing Association Case Competition
MKTG 498H Honors Thesis
"""

# 3. Parse the list to get course keys (Subject + Catalog Number)
course_list_keys = []
for line in raw_courses_text.split('\n'):
    line = line.strip()
    if not line:
        continue
    
    # Extract the first two words (e.g. "ECON" and "150C1")
    # Handling cases like OSCM 471/571 by splitting on slashes as well
    parts = line.split()
    if len(parts) >= 2:
        subject = parts[0].strip()
        catalog_nums = parts[1].strip()
        
        # If there is a slash (e.g. 471/571), grab both numbers
        for cat_num in catalog_nums.split('/'):
            # clean up any trailing symbols attached to the catalog number
            clean_cat = re.sub(r'[^A-Za-z0-9-]', '', cat_num) 
            course_list_keys.append((subject, clean_cat))

# 4. Filter the dataframe based on matching keys
def is_in_list(row):
    # Ensure they are strings before comparing
    subj = str(row.get('Subject code', '')).strip()
    cat = str(row.get('Catalog Number', '')).strip()
    return (subj, cat) in course_list_keys

filtered_df = catalog_df[catalog_df.apply(is_in_list, axis=1)]

# 5. Save the output to a new file
output_file = 'extracted_courses.csv'
filtered_df.to_csv(output_file, index=False)

print(f"Extracted {len(filtered_df)} courses out of {len(catalog_df)} total.")
print(f"File successfully saved as '{output_file}'.")

Extracted 200 courses out of 17849 total.
File successfully saved as 'extracted_courses.csv'.
